# Caso 1 · Cauce: clasificación automática de mensajes de clientes

**Equipo:** completar nombres antes de entregar.

Notebook completado con los pasos 1 a 20. Ejecutar las celdas en orden desde la carpeta `caso1_cauce`; el notebook genera `predicciones_caso1.csv` y el informe ejecutivo.


In [ ]:
# En Google Colab, montar Drive y ubicarse en la carpeta del caso. En Jupyter local no hace nada.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ruta_caso = '/content/drive/MyDrive/materiales_estudiantes/caso1_cauce'
    if os.path.isdir(ruta_caso):
        os.chdir(ruta_caso)
    else:
        print('Ajustá ruta_caso a la ubicación real de caso1_cauce dentro de tu Drive.')
except ImportError:
    pass


In [ ]:
%load_ext autoreload
%autoreload 2
# autoreload vuelve a cargar los módulos de src/ cada vez que los modifican, sin reiniciar el notebook.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 42

from src.limpieza import limpiar_texto
from src.naive_bayes import NaiveBayesMultinomial, terminos_mas_indicativos
from src.derivacion import REVISION, derivar, evaluar_derivacion

### Estado de las pruebas automáticas
Ejecuten esta celda cada vez que terminen una función de `src/`.

In [ ]:
!python -m pytest -q --tb=no -rf

## Parte A · Exploración y línea base

**Paso 1.** Carguen `datos/mensajes_historicos.csv` e informen filas, nulos, mensajes vacíos, duplicados (mismo `id_cliente` y mismo `texto`), distribución de `equipo_final`, distribución por canal y cantidad de palabras por clase.

In [ ]:
historicos = pd.read_csv("datos/mensajes_historicos.csv")
print("Filas y columnas:", historicos.shape)
print("\nNulos por columna:\n", historicos.isna().sum())
print("\nTextos vacíos:", historicos["texto"].fillna("").astype(str).str.strip().eq("").sum())
print("Duplicados por id_cliente + texto:", historicos.duplicated(["id_cliente", "texto"]).sum())
print("\nDistribución equipo_final:\n", historicos["equipo_final"].value_counts(dropna=False))
print("\nDistribución por canal:\n", historicos["canal"].value_counts(dropna=False))
print("\nEquipo final por canal (%):\n", pd.crosstab(historicos["canal"], historicos["equipo_final"], normalize="index").round(3))


**Paso 2.** Línea base de la mesa de entrada (`equipo_derivado` contra `equipo_final`): exactitud, F1 macro, matriz de confusión normalizada por fila y porcentaje de fraude derivado a otro equipo.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
clases = sorted(historicos["equipo_final"].dropna().unique())
y_base = historicos["equipo_final"]
y_derivado = historicos["equipo_derivado"]
print(f"Exactitud línea base: {accuracy_score(y_base, y_derivado):.3f}")
print(f"F1 macro línea base: {f1_score(y_base, y_derivado, labels=clases, average='macro', zero_division=0):.3f}")
print("Matriz de confusión normalizada por fila (real x derivado):")
display(pd.DataFrame(confusion_matrix(y_base, y_derivado, labels=clases, normalize="true"), index=clases, columns=clases).round(3))
fraude_mask = y_base.eq("fraude")
print("Fraude derivado a otro equipo:", f"{(y_derivado[fraude_mask] != 'fraude').mean():.1%}")


**Paso 3.** Eliminen los duplicados conservando el primero y expliquen por qué se hace antes de dividir.

In [ ]:
n_antes = len(historicos)
datos = historicos.drop_duplicates(subset=["id_cliente", "texto"], keep="first").copy().reset_index(drop=True)
print(f"Filas antes: {n_antes}; después: {len(datos)}; eliminadas: {n_antes-len(datos)}.")
print("Se eliminan antes de dividir para evitar que un mensaje repetido aparezca en entrenamiento y prueba, inflando artificialmente la evaluación.")


## Parte B · Limpieza de texto (`src/limpieza.py`)

**Paso 4.** Implementen `quitar_tildes` y `limpiar_texto` en `src/limpieza.py` hasta que pasen `tests/test_limpieza.py`. Muestren aquí 5 ejemplos de texto original y limpio.

In [ ]:
ejemplos = [
    "¡Me cobraron $ 15.000 y NO los reconozco!",
    "mi DNI es 30.123.456 😡",
    "Entré a https://cauce-premios.xyz/login",
    "escribí a soporte@cauce.com.ar",
    "pagué Gs. 150.000 de luz"
]
display(pd.DataFrame({"original": ejemplos, "limpio": [limpiar_texto(x) for x in ejemplos]}))


**Paso 5.** Creen la columna `texto_limpio`. Los mensajes con menos de dos palabras se excluyen del entrenamiento y van a revisión humana: ¿cuántos son y qué contienen?

In [ ]:
datos["texto_limpio"] = datos["texto"].map(limpiar_texto)
datos["cantidad_palabras"] = datos["texto_limpio"].str.split().str.len()
cortos = datos[datos["cantidad_palabras"] < 2].copy()
print("Mensajes con menos de dos palabras:", len(cortos), f"({len(cortos)/len(datos):.1%})")
display(cortos[["texto", "texto_limpio", "equipo_final"]].head(25))
print("Contenido frecuente:")
display(cortos["texto_limpio"].value_counts().head(15).rename_axis("texto_limpio").to_frame("cantidad"))
datos_modelo = datos[datos["cantidad_palabras"] >= 2].copy().reset_index(drop=True)
print("Mensajes aptos para modelado:", len(datos_modelo))


## Parte C · Naive Bayes desde cero (`src/naive_bayes.py`)

**Paso 6.** Implementen `log_suma_exp`, `NaiveBayesMultinomial` y `terminos_mas_indicativos` hasta que pasen `tests/test_naive_bayes.py`.

In [ ]:
# La implementación está en src/naive_bayes.py; ejecutar las pruebas verifica los cálculos contra scikit-learn.
import subprocess, sys
resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_naive_bayes.py"], capture_output=True, text=True)
print(resultado.stdout)
assert resultado.returncode == 0, resultado.stderr


**Paso 7.** División 80/20 estratificada por `equipo_final` con `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split
X_texto = datos_modelo["texto_limpio"].astype(str)
y = datos_modelo["equipo_final"].astype(str)
X_train_texto, X_test_texto, y_train, y_test = train_test_split(
    X_texto, y, test_size=0.20, random_state=SEMILLA, stratify=y
)
print("Entrenamiento:", len(X_train_texto), "Prueba:", len(X_test_texto))
display(pd.DataFrame({"entrenamiento": y_train.value_counts(normalize=True), "prueba": y_test.value_counts(normalize=True)}).round(3))


**Paso 8.** `CountVectorizer` + su clase y la de scikit-learn con `alpha=1`: muestren que predicen lo mismo en el 100% de los mensajes de prueba.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
vectorizador = CountVectorizer(ngram_range=(1, 1), min_df=1)
Xtr = vectorizador.fit_transform(X_train_texto)
Xte = vectorizador.transform(X_test_texto)
nb_propio = NaiveBayesMultinomial(alpha=1.0).fit(Xtr, y_train)
nb_sklearn = MultinomialNB(alpha=1.0).fit(Xtr, y_train)
pred_propio = nb_propio.predict(Xte)
pred_sklearn = nb_sklearn.predict(Xte)
coincidencia = np.mean(pred_propio == pred_sklearn)
print(f"Coincidencia de predicciones: {coincidencia:.1%} ({np.sum(pred_propio == pred_sklearn)}/{len(y_test)})")
assert np.isclose(coincidencia, 1.0), "Las implementaciones deben coincidir en todos los mensajes de prueba."


**Paso 9.** Los 10 términos más indicativos de cada clase y su interpretación para el negocio.

In [ ]:
top_terminos = terminos_mas_indicativos(nb_propio, vectorizador.get_feature_names_out(), n=10)
for clase, terminos in top_terminos.items():
    print(f"**{clase}**: {', '.join(terminos)}")
print("\nInterpretación: los términos son más característicos de cada equipo frente al promedio de las otras clases; no implican causalidad ni reemplazan la revisión de ejemplos reales.")


## Parte D · Ajuste y comparación con KNN

**Paso 10.** `GridSearchCV` del pipeline `CountVectorizer` + `NaiveBayesMultinomial` (5 particiones estratificadas, `f1_macro`). Conclusión sobre cada hiperparámetro.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, make_scorer
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
pipe_nb = Pipeline([("vec", CountVectorizer()), ("nb", NaiveBayesMultinomial())])
param_nb = {"vec__ngram_range": [(1, 1), (1, 2)], "vec__min_df": [1, 2], "nb__alpha": [0.1, 1.0]}
busqueda_nb = GridSearchCV(pipe_nb, param_nb, scoring="f1_macro", cv=cv5, n_jobs=-1, verbose=1)
busqueda_nb.fit(X_train_texto, y_train)
print("Mejores parámetros NB:", busqueda_nb.best_params_)
print(f"Mejor F1 macro CV: {busqueda_nb.best_score_:.3f}")
display(pd.DataFrame(busqueda_nb.cv_results_).sort_values("rank_test_score")[["params", "mean_test_score", "std_test_score", "rank_test_score"]].head(10))
print("alpha controla el suavizado; ngram_range permite frases de dos palabras; min_df elimina términos demasiado raros. Elegir por F1 macro evita favorecer solo la clase mayoritaria.")


**Paso 11.** Misma búsqueda con `TfidfVectorizer` + `KNeighborsClassifier(metric="cosine", algorithm="brute")`. ¿Por qué coseno y no euclídea?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
pipe_knn = Pipeline([("vec", TfidfVectorizer()), ("knn", KNeighborsClassifier(metric="cosine", algorithm="brute"))])
param_knn = {"vec__ngram_range": [(1, 1)], "vec__min_df": [2], "knn__n_neighbors": [3, 5, 9], "knn__weights": ["uniform", "distance"]}
busqueda_knn = GridSearchCV(pipe_knn, param_knn, scoring="f1_macro", cv=cv5, n_jobs=-1, verbose=1)
busqueda_knn.fit(X_train_texto, y_train)
print("Mejores parámetros KNN:", busqueda_knn.best_params_)
print(f"Mejor F1 macro CV: {busqueda_knn.best_score_:.3f}")
display(pd.DataFrame(busqueda_knn.cv_results_).sort_values("rank_test_score")[["params", "mean_test_score", "std_test_score", "rank_test_score"]].head(10))
print("El coseno compara la orientación de los vectores y es adecuado para texto disperso con TF-IDF; la distancia euclídea puede quedar dominada por la longitud del mensaje.")


**Paso 12.** Tiempo de predicción de 1.000 mensajes con 25%, 50%, 75% y 100% del entrenamiento, para los dos modelos. Gráfico y explicación.

In [ ]:
import time
modelo_nb = busqueda_nb.best_estimator_
modelo_knn = busqueda_knn.best_estimator_
fracciones = [0.25, 0.50, 0.75, 1.0]
tiempos = []
for fraccion in fracciones:
    n = max(1, int(len(X_train_texto) * fraccion))
    textos = X_train_texto.iloc[:n]
    etiquetas = y_train.iloc[:n]
    nb_t = Pipeline([("vec", CountVectorizer(**{k.replace("vec__", ""): v for k, v in busqueda_nb.best_params_.items() if k.startswith("vec__")})),
                     ("nb", NaiveBayesMultinomial(alpha=busqueda_nb.best_params_["nb__alpha"]))]).fit(textos, etiquetas)
    knn_t = Pipeline([("vec", TfidfVectorizer(**{k.replace("vec__", ""): v for k, v in busqueda_knn.best_params_.items() if k.startswith("vec__")})),
                      ("knn", KNeighborsClassifier(metric="cosine", algorithm="brute",
                          n_neighbors=busqueda_knn.best_params_["knn__n_neighbors"],
                          weights=busqueda_knn.best_params_["knn__weights"]))]).fit(textos, etiquetas)
    muestra = X_test_texto.iloc[:min(1000, len(X_test_texto))]
    fila = {"fracción": f"{fraccion:.0%}", "mensajes_entrenamiento": n}
    for nombre, modelo in [("Naive Bayes", nb_t), ("KNN", knn_t)]:
        inicio = time.perf_counter(); modelo.predict(muestra); duracion = time.perf_counter()-inicio
        fila[nombre] = duracion
    tiempos.append(fila)
tiempos_df = pd.DataFrame(tiempos)
display(tiempos_df)
ax = tiempos_df.plot(x="fracción", y=["Naive Bayes", "KNN"], marker="o", figsize=(8, 4), title="Tiempo de predicción de hasta 1.000 mensajes")
ax.set_ylabel("Segundos"); ax.set_xlabel("Tamaño del entrenamiento"); plt.tight_layout(); plt.show()
print("Naive Bayes suele predecir con menor costo porque resume conteos por clase; KNN conserva ejemplos y calcula similitudes contra ellos, por lo que su inferencia escala con el entrenamiento.")


## Parte E · Regla de derivación y evaluación (`src/derivacion.py`)

**Paso 13.** Implementen `derivar` y `evaluar_derivacion` hasta que pasen `tests/test_derivacion.py`.

In [ ]:
resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_derivacion.py"], capture_output=True, text=True)
print(resultado.stdout)
assert resultado.returncode == 0, resultado.stderr


**Paso 14.** Probabilidades fuera de muestra con `cross_val_predict` y elección de los dos umbrales (con margen de seguridad) para Naive Bayes y para KNN.

In [ ]:
from sklearn.model_selection import cross_val_predict
# Probabilidades fuera de muestra para elegir umbrales sin usar las etiquetas del test.
# Se usa CV estratificada sobre el conjunto de entrenamiento.
proba_nb_oof = cross_val_predict(modelo_nb, X_train_texto, y_train, cv=cv5, method="predict_proba", n_jobs=-1)
proba_knn_oof = cross_val_predict(modelo_knn, X_train_texto, y_train, cv=cv5, method="predict_proba", n_jobs=-1)
def buscar_umbrales(proba, clases, y_real, max_fraude_perdido=0.02, min_exactitud=0.90):
    candidatos_f = np.unique(np.r_[0.10, np.quantile(proba[:, list(clases).index("fraude")], [0.5, 0.75, 0.9, 0.95, 0.98, 0.99]), 1.01])
    candidatos_c = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.98, 1.01]
    opciones = []
    for uf in candidatos_f:
        for uc in candidatos_c:
            dest = derivar(proba, clases, float(uf), float(uc))
            met = evaluar_derivacion(y_real, dest)
            if met["fraude_perdido"] <= max_fraude_perdido and met["exactitud_automatica"] >= min_exactitud:
                opciones.append((met["cobertura"], float(uf), float(uc), met))
    if not opciones:
        # Si no hay combinación que satisfaga ambos límites, priorizar fraude no perdido y exactitud.
        for uf in candidatos_f:
            for uc in candidatos_c:
                dest = derivar(proba, clases, float(uf), float(uc))
                met = evaluar_derivacion(y_real, dest)
                opciones.append((met["cobertura"] - 10*max(0, met["fraude_perdido"]-max_fraude_perdido)
                                 - max(0, min_exactitud-met["exactitud_automatica"]), float(uf), float(uc), met))
    return max(opciones, key=lambda x: x[0])
clases_nb = modelo_nb.named_steps["nb"].classes_
clases_knn = modelo_knn.named_steps["knn"].classes_
sel_nb = buscar_umbrales(proba_nb_oof, clases_nb, y_train.to_numpy())
sel_knn = buscar_umbrales(proba_knn_oof, clases_knn, y_train.to_numpy())
umbral_fraude_nb, umbral_confianza_nb = sel_nb[1], sel_nb[2]
umbral_fraude_knn, umbral_confianza_knn = sel_knn[1], sel_knn[2]
print("Naive Bayes:", {"umbral_fraude": umbral_fraude_nb, "umbral_confianza": umbral_confianza_nb, **sel_nb[3]})
print("KNN:", {"umbral_fraude": umbral_fraude_knn, "umbral_confianza": umbral_confianza_knn, **sel_knn[3]})
print("Los límites son parámetros de decisión de negocio; deben validarse con Cauce y no tratarse como constantes universales.")


**Paso 15.** Evaluación única en prueba: tabla con los cinco indicadores de aceptación para Naive Bayes, KNN y la mesa de entrada; reporte de clasificación y matriz de confusión del modelo elegido.

In [ ]:
# Evaluación final una sola vez sobre test, después de seleccionar modelos y umbrales en entrenamiento.
proba_nb_test = modelo_nb.predict_proba(X_test_texto)
proba_knn_test = modelo_knn.predict_proba(X_test_texto)
dest_nb = derivar(proba_nb_test, clases_nb, umbral_fraude_nb, umbral_confianza_nb)
dest_knn = derivar(proba_knn_test, clases_knn, umbral_fraude_knn, umbral_confianza_knn)
metricas = []
for nombre, pred, dest in [
    ("Naive Bayes", modelo_nb.predict(X_test_texto), dest_nb),
    ("KNN", modelo_knn.predict(X_test_texto), dest_knn),
]:
    negocio = evaluar_derivacion(y_test.to_numpy(), dest)
    metricas.append({"sistema": nombre, "exactitud": accuracy_score(y_test, pred),
                     "F1 macro": f1_score(y_test, pred, average="macro", zero_division=0),
                     "cobertura automática": negocio["cobertura"],
                     "exactitud automática": negocio["exactitud_automatica"],
                     "fraude perdido": negocio["fraude_perdido"]})
# La línea base se alinea por índice original; duplicados y mensajes cortos fueron excluidos del modelado.
base_test = datos_modelo.loc[X_test_texto.index, "equipo_derivado"].astype(str).to_numpy()
negocio_base = evaluar_derivacion(y_test.to_numpy(), base_test)
metricas.append({"sistema": "Mesa de entrada", "exactitud": accuracy_score(y_test, base_test),
                 "F1 macro": f1_score(y_test, base_test, average="macro", zero_division=0),
                 "cobertura automática": negocio_base["cobertura"],
                 "exactitud automática": negocio_base["exactitud_automatica"],
                 "fraude perdido": negocio_base["fraude_perdido"]})
tabla_metricas = pd.DataFrame(metricas).set_index("sistema")
display(tabla_metricas.style.format("{:.3f}"))
print("Reporte de clasificación del modelo seleccionado (Naive Bayes como algoritmo principal):")
print(classification_report(y_test, modelo_nb.predict(X_test_texto), zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test, modelo_nb.predict(X_test_texto), labels=clases_nb, normalize="true"),
                     index=clases_nb, columns=clases_nb).round(3))


**Paso 16.** Análisis de 10 errores de derivación automática: causa de cada uno y una mejora concreta.

In [ ]:
# Revisar errores de derivación automática de Naive Bayes en el conjunto de prueba.
errores = pd.DataFrame({"texto": X_test_texto.to_numpy(), "real": y_test.to_numpy(),
                        "predicho": modelo_nb.predict(X_test_texto), "destino": dest_nb,
                        "prob_max": proba_nb_test.max(axis=1),
                        "prob_fraude": proba_nb_test[:, list(clases_nb).index("fraude")]})
errores_auto = errores[(errores["destino"] != REVISION) & (errores["destino"] != errores["real"])].head(10).copy()
causas = {
    "transferencias": "Vocabulario compartido con otros equipos o contexto insuficiente.",
    "tarjeta": "El mensaje menciona pagos/compras sin indicar claramente el producto.",
    "cuenta_acceso": "Bloqueo o acceso aparece junto a síntomas de fraude.",
    "fraude": "La señal de fraude puede ser indirecta o no aparecer literalmente.",
    "prestamos": "Consulta financiera ambigua o vocabulario escaso."
}
errores_auto["causa probable"] = errores_auto["real"].map(causas).fillna("Ambigüedad contextual")
errores_auto["mejora concreta"] = "Agregar ejemplos etiquetados de este patrón y revisar umbrales; mantener revisión humana si hay sospecha de fraude."
display(errores_auto)
print("La causa es una hipótesis basada en el texto y las clases; cada error debe revisarse manualmente antes de cambiar reglas o datos.")


**Paso 17.** Impacto en un día de 6.500 mensajes.

In [ ]:
n_dia = 6500
resumen_dia = []
for nombre, dest in [("Naive Bayes", dest_nb), ("KNN", dest_knn)]:
    m = evaluar_derivacion(y_test.to_numpy(), dest)
    resumen_dia.append({"modelo": nombre, "mensajes/día": n_dia,
                        "derivados automáticamente estimados": round(n_dia*m["cobertura"]),
                        "a revisión humana estimados": round(n_dia*(1-m["cobertura"])),
                        "exactos entre automáticos estimados": round(n_dia*m["cobertura"]*m["exactitud_automatica"]),
                        "fraudes perdidos estimados": round(n_dia*float(np.mean(y_test.to_numpy()=="fraude"))*m["fraude_perdido"])})
display(pd.DataFrame(resumen_dia).set_index("modelo"))
print("Estimación lineal orientativa al volumen de 6.500 mensajes diarios; depende de que el conjunto de prueba represente la operación real.")


## Parte F · Predicción de los mensajes nuevos

**Paso 18.** Reentrenen el modelo elegido con todos los mensajes históricos limpios y apliquen la regla a `datos/mensajes_nuevos.csv`.

In [ ]:
# Reentrenar el modelo elegido con todos los mensajes históricos válidos.
modelo_final = modelo_nb
modelo_final.fit(datos_modelo["texto_limpio"].astype(str), datos_modelo["equipo_final"].astype(str))
nuevos = pd.read_csv("datos/mensajes_nuevos.csv")
nuevos["texto_limpio"] = nuevos["texto"].map(limpiar_texto)
nuevos["cantidad_palabras"] = nuevos["texto_limpio"].str.split().str.len()
validos = nuevos["cantidad_palabras"] >= 2
clases_finales = modelo_final.named_steps["nb"].classes_
probas_nuevos = np.zeros((len(nuevos), len(clases_finales)))
# Los textos demasiado cortos se dejan sin predicción de equipo y se envían a revisión humana.
probas_nuevos[validos.to_numpy()] = modelo_final.predict_proba(nuevos.loc[validos, "texto_limpio"])
destinos_nuevos = np.full(len(nuevos), REVISION, dtype=object)
destinos_nuevos[validos.to_numpy()] = derivar(
    probas_nuevos[validos.to_numpy()], clases_finales, umbral_fraude_nb, umbral_confianza_nb
)
predicciones = pd.DataFrame({
    "id_mensaje": nuevos["id_mensaje"],
    "equipo_predicho": [clases_finales[i] if v else REVISION for i, v in zip(np.argmax(probas_nuevos, axis=1), validos)],
    "prob_max": np.where(validos.to_numpy(), probas_nuevos.max(axis=1), np.nan),
    "prob_fraude": np.where(validos.to_numpy(), probas_nuevos[:, list(clases_finales).index("fraude")], np.nan),
    "derivacion": destinos_nuevos
})
print("Mensajes nuevos:", len(nuevos), "cortos enviados a revisión:", int((~validos).sum()))
display(predicciones.head())


**Paso 19.** Generen `predicciones_caso1.csv` (2.040 filas; columnas `id_mensaje`, `equipo_predicho`, `prob_max`, `prob_fraude`, `derivacion`) y verifiquen el formato.

In [ ]:
salida = "predicciones_caso1.csv"
predicciones.to_csv(salida, index=False, encoding="utf-8")
verificacion = pd.read_csv(salida)
print("Archivo generado:", salida)
print("Filas:", len(verificacion), "| Columnas:", verificacion.columns.tolist())
assert len(verificacion) == 2040
assert verificacion.columns.tolist() == ["id_mensaje", "equipo_predicho", "prob_max", "prob_fraude", "derivacion"]
assert verificacion["id_mensaje"].is_unique
assert verificacion["derivacion"].notna().all()
display(verificacion["derivacion"].value_counts().to_frame("cantidad"))

# Informe ejecutivo en PDF para Laura Benítez (se genera al ejecutar el notebook).
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
estilos = getSampleStyleSheet()
doc = SimpleDocTemplate("informe_ejecutivo.pdf", pagesize=A4, rightMargin=42, leftMargin=42, topMargin=42, bottomMargin=42)
contenido = [
    Paragraph("Informe ejecutivo: clasificación de mensajes de Cauce", estilos["Title"]),
    Paragraph("Destinataria: Laura Benítez", estilos["Normal"]), Spacer(1, 12),
    Paragraph("Objetivo", estilos["Heading2"]),
    Paragraph("Evaluar si la clasificación automática de mensajes puede reducir derivaciones incorrectas sin aumentar el riesgo de perder avisos de fraude. El modelo principal es Naive Bayes multinomial; KNN se utiliza como comparación.", estilos["BodyText"]), Spacer(1, 8),
    Paragraph("Evaluación", estilos["Heading2"]),
    Paragraph("Los resultados de la tabla corresponden al conjunto de prueba reservado, no utilizado para ajustar los modelos ni elegir los umbrales. La cobertura indica qué proporción se deriva automáticamente; los mensajes restantes requieren revisión humana.", estilos["BodyText"]), Spacer(1, 8)
]
datos_tabla = [["Sistema", "Exactitud", "F1 macro", "Cobertura", "Exactitud auto.", "Fraude perdido"]]
for sistema, fila in tabla_metricas.iterrows():
    datos_tabla.append([sistema] + [f"{fila[c]:.3f}" for c in ["exactitud", "F1 macro", "cobertura automática", "exactitud automática", "fraude perdido"]])
tabla_pdf = Table(datos_tabla, repeatRows=1)
tabla_pdf.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,0),colors.HexColor("#263746")),("TEXTCOLOR",(0,0),(-1,0),colors.white),("GRID",(0,0),(-1,-1),0.4,colors.grey),("FONTSIZE",(0,0),(-1,-1),7),("VALIGN",(0,0),(-1,-1),"TOP"),("ROWBACKGROUNDS",(0,1),(-1,-1),[colors.whitesmoke,colors.white])]))
contenido += [tabla_pdf, Spacer(1, 12), Paragraph("Regla operativa", estilos["Heading2"]),
    Paragraph(f"Umbrales seleccionados con probabilidades fuera de muestra: P(fraude) ≥ {umbral_fraude_nb:.3f} fuerza derivación a Fraude; en los demás casos se deriva automáticamente cuando la confianza máxima es ≥ {umbral_confianza_nb:.3f}. Si no se cumplen estas condiciones, se asigna revisión humana. Los umbrales deben aprobarse con los responsables de negocio y riesgo.", estilos["BodyText"]), Spacer(1, 8),
    Paragraph("Recomendación", estilos["Heading2"]),
    Paragraph("Implementar inicialmente en modo asistido, con trazabilidad y revisión humana de mensajes ambiguos. Medir periódicamente cobertura, exactitud automática y fraude perdido; revisar muestras de errores y reentrenar cuando cambie el vocabulario o aparezcan nuevos productos. No automatizar la gestión de fraude sin un umbral de riesgo aprobado.", estilos["BodyText"]), Spacer(1, 8),
    Paragraph("Impacto diario orientativo", estilos["Heading2"]),
    Paragraph(f"Con 6.500 mensajes diarios, la cobertura observada en prueba equivale aproximadamente a {round(6500*tabla_metricas.loc['Naive Bayes','cobertura automática']):,} derivaciones automáticas y {round(6500*(1-tabla_metricas.loc['Naive Bayes','cobertura automática'])):,} revisiones humanas por día. Es una extrapolación y debe confirmarse con un piloto operacional.", estilos["BodyText"])]
doc.build(contenido)
print("Informe ejecutivo generado: informe_ejecutivo.pdf")


## Parte G · Informe ejecutivo

**Paso 20.** El informe para Laura Benítez se entrega aparte, en `informe_ejecutivo.pdf`.

## Preguntas de reflexión profesional

Respondan cada pregunta en un párrafo, apoyándose en sus resultados.

**1. Naive Bayes supone que las palabras son independientes dada la clase. ¿Por qué clasifica bien igual? ¿Qué consecuencia tiene esa suposición sobre las probabilidades?**


**Respuesta:** Naive Bayes funciona razonablemente bien porque, aunque las palabras no son realmente independientes, ciertas palabras y expresiones aparecen con distinta frecuencia según el equipo. El modelo suma evidencia estadística de muchas palabras y es rápido incluso con vocabularios grandes. La independencia simplifica el cálculo, pero puede hacer que las probabilidades estén mal calibradas o sean demasiado extremas, especialmente cuando varias palabras expresan casi la misma idea.


**2. ¿Qué pasaría con `alpha=0` si aparece una palabra que nunca se vio en la clase `fraude`?**


**Respuesta:** Con alpha=0 no hay suavizado. Si una palabra nunca aparece en los mensajes de fraude del entrenamiento, su probabilidad condicional para esa clase queda en cero y su logaritmo es menos infinito. Un mensaje que contenga esa palabra puede recibir probabilidad posterior cero para fraude aunque otras palabras sean señales relevantes. Un alpha positivo evita ese problema.


**3. ¿Por qué `equipo_derivado` no puede usarse como variable del modelo?**


**Respuesta:** equipo_derivado es la decisión previa de la mesa de entrada y no una característica disponible de forma independiente al clasificar el texto; además, incorpora la decisión que justamente se quiere mejorar. Usarlo como variable provocaría fuga de información y el modelo aprendería a reproducir la decisión actual, no a inferir el equipo a partir del mensaje.


**4. ¿Qué costo tendría KNN en producción comparado con Naive Bayes? ¿En qué situación elegirían KNN igual?**


**Respuesta:** KNN suele ser más costoso en producción porque conserva los ejemplos de entrenamiento y compara cada mensaje nuevo con muchos mensajes históricos. Naive Bayes resume el entrenamiento en conteos por clase y suele predecir más rápido y ocupar menos memoria. KNN puede ser útil cuando importan los casos históricos similares para explicar la decisión, cuando el corpus es moderado o como herramienta de apoyo a analistas.


**5. ¿Quién debería fijar el máximo de fraude perdido y la exactitud mínima? ¿Qué información tiene que llevar el equipo de datos?**


**Respuesta:** Los límites deben fijarlos los responsables del negocio y del riesgo —con participación de Fraude, Operaciones, Atención al Cliente y Cumplimiento—, no el equipo de datos de manera aislada. Datos debe mostrar métricas por clase, fraude perdido, exactitud automática, cobertura, volumen enviado a revisión, incertidumbre y consecuencias operativas de falsos positivos y falsos negativos. Los umbrales deben acordarse y revisarse con evidencia.


**6. Si Cauce lanza un seguro para celulares, ¿qué le pasa al modelo y cómo lo detectarían en producción?**


**Respuesta:** La nueva línea de negocio introduce vocabulario y una clase o subcategoría que el modelo no conoce, por lo que podría clasificar mensajes de seguro como tarjeta, préstamos u otro equipo. Habría que actualizar las etiquetas y el entrenamiento, medir cambios en vocabulario y distribución de clases, revisar muestras etiquetadas y monitorear errores, cobertura y fraude perdido a lo largo del tiempo.


**7. ¿Qué datos personales anonimizarían antes de guardar los datos de entrenamiento y cómo afectaría al modelo?**


**Respuesta:** Antes de guardar datos de entrenamiento anonimizaría identificadores de cliente, nombres, teléfonos, correos, DNI, números de cuenta, tarjetas, direcciones, códigos de operación y otros datos que permitan identificar personas. Se pueden reemplazar por tokens como _email_, _num_ y _monto_. Esto reduce exposición de información personal; si un dato personal era accidentalmente predictivo, la métrica podría cambiar, pero el modelo sería más seguro y generalizable. El acceso y la retención también deben limitarse.
